# NLP Analysis of Political Manifestos

This notebook analyses political manifestos from the Archelec corpus using NLP techniques.

We study:
- semantic density (linguistic richness)
- semantic similarity (TF-IDF vs embeddings)
- ranking of similar documents

In [ ]:
!pip install pandas numpy matplotlib scikit-learn spacy sentence-transformers tqdm
!python -m spacy download fr_core_news_sm

In [ ]:
import os
import re
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import spacy
from sentence_transformers import SentenceTransformer

## Data Loading

We extract the dataset from the uploaded zip file.

In [ ]:
# NOTE: dataset not included in the repository.
# Place the dataset inside the "data/" folder before running.

import zipfile

zip_path = "/content/legislatives (1).zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

print("Extraction completed")


In [ ]:
folder = "/content/data/text_files/1988/legislatives"

data = []

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        with open(os.path.join(folder, filename), "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        data.append({"id": filename, "text": text})

df = pd.DataFrame(data)

print("Number of documents:", len(df))
df.head()

## Text Preprocessing

We clean the text by removing noise and normalizing it.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-zàâçéèêëîïôûùüÿñæœ\s]', ' ', text)
    return text

df["clean_text"] = df["text"].apply(clean_text)

## Exploratory Data Analysis

We analyse document lengths.

In [ ]:
df["length"] = df["clean_text"].apply(lambda x: len(x.split()))

plt.hist(df["length"], bins=50)
plt.title("Document Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.show()

plt.savefig("length_plot.png")

## Semantic Density

We define semantic density as the ratio of content words to total words.

In [ ]:
nlp = spacy.load("fr_core_news_sm")

def semantic_density(text):
    doc = nlp(text[:1000])
    content = [t for t in doc if t.pos_ in ["NOUN","VERB","ADJ"]]
    return len(content) / len(doc) if len(doc) > 0 else 0

df["density"] = df["clean_text"].apply(semantic_density)

## Density Distribution

In [ ]:
plt.hist(df["density"], bins=50)
plt.title("Semantic Density Distribution")
plt.xlabel("Density")
plt.ylabel("Frequency")
plt.show()

plt.savefig("density_plot.png")

In [ ]:
print("LOW DENSITY EXAMPLE:\n")
print(df.sort_values("density").iloc[0]["text"][:500])

print("\nHIGH DENSITY EXAMPLE:\n")
print(df.sort_values("density", ascending=False).iloc[0]["text"][:500])

## TF-IDF Representation

We compute a lexical baseline using TF-IDF.

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

tfidf_sim = cosine_similarity(tfidf_matrix)

## Embedding-based Representation

We use sentence embeddings to capture semantic relationships.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df["clean_text"].tolist(), show_progress_bar=True)

## Similarity Computation

emb_sim = cosine_similarity(embeddings)

## Ranking Task

We retrieve the most similar documents given a query.

In [ ]:
def get_top_k(sim_matrix, idx, k=5):
    scores = sim_matrix[idx]
    return scores.argsort()[::-1][1:k+1]

In [ ]:
query_idx = 0

print("QUERY:\n")
print(df.iloc[query_idx]["text"][:500])

print("\nTOP 5 SIMILI:\n")

top_docs = get_top_k(emb_sim, query_idx)

for i in top_docs:
    print(f"\n--- Documento {i} ---\n")
    print(df.iloc[i]["text"][:300])

## Model Comparison

In [ ]:
print("TF-IDF:", get_top_k(tfidf_sim, query_idx))
print("EMBEDDINGS:", get_top_k(emb_sim, query_idx))

## Discussion

Semantic density highlights differences in informational content.  
Embedding-based similarity captures deeper relationships than TF-IDF.  

Limitations include OCR noise and simplified linguistic assumptions.

## Conclusion

NLP techniques allow meaningful analysis of political texts, both linguistically and semantically.